# Improved Hallucination Detection Baselines

This notebook upgrades the original baseline evaluation in three ways:

1. It rebuilds the clean ToolACE split directly from the original dataset.
2. It evaluates on a combined benchmark made of `clean + 3 hallucination datasets` from `datasets/`.
3. It trains stronger sample-level detectors on real labels and reports both overall and per-type metrics.

The goal is to benchmark hallucination detection for:
- `tool_output_contradiction`
- `overgeneration`
- `missing_tool`


In [ ]:
# Install additional packages if needed.
%pip install -q lettucedetect scikit-learn transformers matplotlib seaborn


## 1. Imports and setup


In [ ]:
import copy
import json
import random
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

seed = 1241
random.seed(seed)
np.random.seed(seed)

sns.set_theme(style="whitegrid")


## 2. Rebuild the clean ToolACE dataset

The assignment requires the original dataset to be loaded from:

```python
init_dataset = pd.read_json("hf://datasets/Team-ACE/ToolACE/data.json")
```


In [ ]:
init_dataset = pd.read_json("hf://datasets/Team-ACE/ToolACE/data.json")
print(f"Loaded raw ToolACE rows: {len(init_dataset)}")
init_dataset.head(2)


In [ ]:
def tool_responses_to_text(responses):
    parts = []
    for response in responses:
        parts.append(f"{response['name']}: {json.dumps(response['results'], ensure_ascii=False)}")
    return "\n".join(parts)


def parse_one_conversation(conv):
    res = []
    n = len(conv)
    i = 0

    while i < n:
        if conv[i].get("from") != "tool":
            i += 1
            continue

        context = copy.deepcopy(json.loads(conv[i]["value"]))
        entry = {}

        j = i
        while j >= 0 and conv[j].get("from") != "user":
            j -= 1
        if j < 0:
            i += 1
            continue
        entry["query"] = conv[j]["value"]

        j = i + 1
        while j < n:
            role = conv[j].get("from")
            if role == "assistant":
                val = conv[j]["value"]
                if val.startswith("[") and val.endswith("]"):
                    j += 1
                    continue
                break
            if role == "tool":
                context += json.loads(conv[j]["value"])
                i = j
            j += 1

        if j >= n:
            i += 1
            continue

        entry["context"] = tool_responses_to_text(context)
        entry["output"] = conv[j]["value"]
        res.append(entry)
        i += 1

    return res


def extract_tools_list_from_system(system_text: str):
    anchor = "Here is a list of functions"
    start_pos = system_text.find(anchor)
    s = system_text[start_pos:] if start_pos != -1 else system_text

    i0 = s.find("[")
    if i0 == -1:
        return None

    depth = 0
    in_str = False
    quote = ""
    esc = False
    start = None

    for i in range(i0, len(s)):
        ch = s[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == quote:
                in_str = False
            continue

        if ch in ('"', "'"):
            in_str = True
            quote = ch
            continue

        if ch == "[":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "]":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    block = s[start:i + 1]
                    try:
                        obj = json.loads(block)
                    except Exception:
                        return None
                    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj):
                        return obj
                    return None
    return None


In [ ]:
correct_dataset = []

for row in init_dataset.itertuples():
    parsed_conv = parse_one_conversation(row.conversations)
    parsed_tool_list = extract_tools_list_from_system(row.system)

    if parsed_tool_list and parsed_conv:
        tool_meta = [{k: t[k] for k in ("name", "description") if k in t} for t in parsed_tool_list]
        for request in parsed_conv:
            request["context"] = f"{request['context']}\nAvailable tools: {json.dumps(tool_meta, ensure_ascii=False)}"

    correct_dataset.extend(parsed_conv)

print(f"Recovered clean ToolACE-style samples: {len(correct_dataset)}")
pd.DataFrame(correct_dataset).head(3)


## 3. Load the three hallucination datasets and build one benchmark


In [ ]:
DATA_DIR = Path("datasets")
CORRUPTION_FILES = {
    "tool_output_contradiction": DATA_DIR / "tool_output_contradiction_dataset.jsonl",
    "overgeneration": DATA_DIR / "overgeneration_dataset.jsonl",
    "missing_tool": DATA_DIR / "missing_tool_dataset.jsonl",
}


def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


corrupted_datasets = {name: load_jsonl(path) for name, path in CORRUPTION_FILES.items()}
for name, rows in corrupted_datasets.items():
    print(f"{name:28s} -> {len(rows)} rows")


In [ ]:
def is_hallucinated(row):
    return int(len(row.get("hallucination_labels", [])) > 0)


n_clean = len(correct_dataset)
for name, rows in corrupted_datasets.items():
    assert len(rows) == n_clean, f"Length mismatch for {name}: {len(rows)} vs {n_clean}"

benchmark_rows = []

for idx, row in enumerate(correct_dataset):
    benchmark_rows.append({
        "example_id": idx,
        "query": row["query"],
        "context": row["context"],
        "output": row["output"],
        "label": 0,
        "corruption_type": "clean",
        "hallucination_labels": [],
        "meta": {"source": "recovered_clean_toolace"},
    })

for corruption_type, rows in corrupted_datasets.items():
    for idx, row in enumerate(rows):
        benchmark_rows.append({
            "example_id": idx,
            "query": row["query"],
            "context": row["context"],
            "output": row["output"],
            "label": is_hallucinated(row),
            "corruption_type": corruption_type,
            "hallucination_labels": row.get("hallucination_labels", []),
            "meta": row.get("meta", {}),
        })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df["row_id"] = np.arange(len(benchmark_df))

print(f"Combined benchmark size: {len(benchmark_df)}")
display(benchmark_df.groupby(["corruption_type", "label"]).size().rename("count").reset_index())
benchmark_df.head(5)


In [ ]:
# Sanity-check a few aligned rows across clean/corrupted variants.
for idx in [0, 1, 2]:
    subset = benchmark_df[benchmark_df["example_id"] == idx][["corruption_type", "label", "query"]]
    display(subset)


## 4. Grouped train/test split without leakage

Every original ToolACE sample produces four rows in the benchmark:
- one clean example
- one contradiction example
- one overgeneration example
- one missing-tool example

To avoid leakage, we split by `example_id`, not by row.


In [ ]:
all_example_ids = sorted(benchmark_df["example_id"].unique())
train_ids, test_ids = train_test_split(all_example_ids, test_size=0.2, random_state=seed)

train_df = benchmark_df[benchmark_df["example_id"].isin(train_ids)].reset_index(drop=True)
test_df = benchmark_df[benchmark_df["example_id"].isin(test_ids)].reset_index(drop=True)

print(f"Train rows: {len(train_df)}")
print(f"Test rows:  {len(test_df)}")
print(f"Overlap in example_id: {len(set(train_ids) & set(test_ids))}")

display(train_df.groupby(["corruption_type", "label"]).size().rename("train_count").reset_index())
display(test_df.groupby(["corruption_type", "label"]).size().rename("test_count").reset_index())


## 5. Evaluation helpers


In [ ]:
def evaluate_predictions(y_true, y_pred, y_score=None, prefix="model"):
    metrics = {
        "model": prefix,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    metrics["precision"] = p
    metrics["recall"] = r
    metrics["f1_binary"] = f1

    if y_score is not None and len(np.unique(y_true)) > 1:
        metrics["auroc"] = roc_auc_score(y_true, y_score)
        metrics["ap"] = average_precision_score(y_true, y_score)
    else:
        metrics["auroc"] = np.nan
        metrics["ap"] = np.nan

    return metrics


def evaluate_by_type(df, pred_col, score_col=None):
    rows = []
    for corruption_type, sub in df.groupby("corruption_type"):
        y_true = sub["label"].to_numpy()
        y_pred = sub[pred_col].to_numpy()
        y_score = sub[score_col].to_numpy() if score_col and score_col in sub.columns else None
        rows.append(evaluate_predictions(y_true, y_pred, y_score=y_score, prefix=corruption_type))
    return pd.DataFrame(rows)


## 6. Baseline A: tool-aware lexical consistency verifier

This baseline is designed specifically for tool-augmented hallucinations.
It uses only text features derived from:
- answer vs tool-output overlap
- numeric/date/percentage grounding
- named entity / title grounding proxies
- actions that may require unavailable tools


In [ ]:
ACTION_PATTERNS = {
    "email": [r"\bemail\b", r"\bmail\b", r"\bsend (an )?email\b"],
    "calendar": [r"\bcalendar\b", r"\bschedule\b", r"\bbook\b", r"\bset up (a )?meeting\b"],
    "phone": [r"\bcall\b", r"\bphone\b", r"\bdial\b"],
    "message": [r"\bslack\b", r"\bmessage\b", r"\bping\b", r"\btext\b"],
}


def split_context_and_tools(context):
    marker = "Available tools: "
    if marker in context:
        tool_output, tools_json = context.split(marker, 1)
        return tool_output.strip(), tools_json.strip()
    return context.strip(), ""


def parse_available_tools(context):
    _, tools_json = split_context_and_tools(context)
    if not tools_json:
        return []
    try:
        parsed = json.loads(tools_json)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass
    return []


def normalize_text(text):
    return re.sub(r"\s+", " ", text.lower()).strip()


def content_tokens(text):
    toks = re.findall(r"[A-Za-z0-9_./%:-]+", text.lower())
    return [t for t in toks if len(t) >= 3]


def extract_percentages(text):
    return re.findall(r"[+-]?\d+(?:\.\d+)?%", text)


def extract_dates(text):
    pats = [
        r"\b\d{4}-\d{2}-\d{2}\b",
        r"\b\d{4}/\d{2}/\d{2}\b",
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2},\s+\d{4}\b",
    ]
    vals = []
    for pat in pats:
        vals.extend(re.findall(pat, text))
    return vals


def extract_numbers(text):
    return re.findall(r"\b\d+(?:\.\d+)?\b", text)


def extract_capitalized_phrases(text):
    return re.findall(r"\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})\b", text)


def fraction_missing(values, context_text):
    if not values:
        return 0.0
    context_norm = normalize_text(context_text)
    missing = 0
    for value in values:
        if normalize_text(value) not in context_norm:
            missing += 1
    return missing / len(values)


def tool_affordance_mismatch(output, available_tools):
    tool_blob = normalize_text(" ".join(
        f"{tool.get('name', '')} {tool.get('description', '')}" for tool in available_tools
    ))
    flags = {}
    for affordance, patterns in ACTION_PATTERNS.items():
        mentions_action = any(re.search(pat, output, flags=re.IGNORECASE) for pat in patterns)
        supported = affordance in tool_blob
        flags[f"action_{affordance}_mentioned"] = int(mentions_action)
        flags[f"action_{affordance}_unsupported"] = int(mentions_action and not supported)
    return flags


def lexical_features(row):
    tool_output, _ = split_context_and_tools(row["context"])
    available_tools = parse_available_tools(row["context"])
    output = row["output"]

    out_tokens = content_tokens(output)
    ctx_token_set = set(content_tokens(tool_output))
    overlap = np.mean([token in ctx_token_set for token in out_tokens]) if out_tokens else 0.0

    percents = extract_percentages(output)
    dates = extract_dates(output)
    numbers = extract_numbers(output)
    caps = extract_capitalized_phrases(output)

    feats = {
        "token_overlap_ratio": overlap,
        "pct_missing_percentages": fraction_missing(percents, tool_output),
        "pct_missing_dates": fraction_missing(dates, tool_output),
        "pct_missing_numbers": fraction_missing(numbers, tool_output),
        "pct_missing_capitalized": fraction_missing(caps, tool_output),
        "answer_len_chars": len(output),
        "answer_len_tokens": len(out_tokens),
        "num_percentages": len(percents),
        "num_dates": len(dates),
        "num_numbers": len(numbers),
        "num_capitalized_phrases": len(caps),
    }
    feats.update(tool_affordance_mismatch(output, available_tools))
    return feats


In [ ]:
lex_train = pd.DataFrame([lexical_features(row) for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Lexical train")])
lex_test = pd.DataFrame([lexical_features(row) for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Lexical test")])

y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

lex_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
])
lex_clf.fit(lex_train, y_train)

lex_test_pred = lex_clf.predict(lex_test)
lex_test_score = lex_clf.predict_proba(lex_test)[:, 1]

test_df_lex = test_df.copy()
test_df_lex["lex_pred"] = lex_test_pred
test_df_lex["lex_score"] = lex_test_score

lex_overall = pd.DataFrame([evaluate_predictions(y_test, lex_test_pred, lex_test_score, prefix="lexical_verifier")])
lex_by_type = evaluate_by_type(test_df_lex, pred_col="lex_pred", score_col="lex_score")

display(lex_overall)
display(lex_by_type)


## 7. Baseline B: improved LettuceDetect

Instead of using only `len(spans) > 0`, we derive richer sample-level features from hallucinated spans and train a supervised classifier on real labels.


In [ ]:
from lettucedetect.models.inference import HallucinationDetector

MODEL_PATH = "KRLabsOrg/lettucedect-base-modernbert-en-v1"
print(f"Loading LettuceDetect model: {MODEL_PATH}")
lettuce_detector = HallucinationDetector(method="transformer", model_path=MODEL_PATH)
print("Model loaded.")


In [ ]:
def run_lettuce(row):
    spans = lettuce_detector.predict(
        context=[row["context"]],
        question=row["query"],
        answer=row["output"],
        output_format="spans",
    )

    scores = [float(span.get("score", 0.0)) for span in spans]
    span_lengths = [max(0, int(span.get("end", 0)) - int(span.get("start", 0))) for span in spans]
    answer_len = max(1, len(row["output"]))

    return {
        "num_spans": len(spans),
        "max_score": max(scores) if scores else 0.0,
        "mean_score": float(np.mean(scores)) if scores else 0.0,
        "top2_mean_score": float(np.mean(sorted(scores, reverse=True)[:2])) if scores else 0.0,
        "sum_span_chars": float(sum(span_lengths)),
        "span_char_fraction": float(sum(span_lengths) / answer_len),
    }


In [ ]:
LETTUCE_TRAIN_LIMIT = None
LETTUCE_TEST_LIMIT = None

lettuce_train_rows = train_df if LETTUCE_TRAIN_LIMIT is None else train_df.iloc[:LETTUCE_TRAIN_LIMIT]
lettuce_test_rows = test_df if LETTUCE_TEST_LIMIT is None else test_df.iloc[:LETTUCE_TEST_LIMIT]

lettuce_train_feats = []
for _, row in tqdm(lettuce_train_rows.iterrows(), total=len(lettuce_train_rows), desc="Lettuce train"):
    try:
        lettuce_train_feats.append(run_lettuce(row))
    except Exception as exc:
        lettuce_train_feats.append({
            "num_spans": 0,
            "max_score": 0.0,
            "mean_score": 0.0,
            "top2_mean_score": 0.0,
            "sum_span_chars": 0.0,
            "span_char_fraction": 0.0,
            "error": str(exc),
        })

lettuce_test_feats = []
for _, row in tqdm(lettuce_test_rows.iterrows(), total=len(lettuce_test_rows), desc="Lettuce test"):
    try:
        lettuce_test_feats.append(run_lettuce(row))
    except Exception as exc:
        lettuce_test_feats.append({
            "num_spans": 0,
            "max_score": 0.0,
            "mean_score": 0.0,
            "top2_mean_score": 0.0,
            "sum_span_chars": 0.0,
            "span_char_fraction": 0.0,
            "error": str(exc),
        })

lettuce_train_X = pd.DataFrame(lettuce_train_feats).drop(columns=["error"], errors="ignore")
lettuce_test_X = pd.DataFrame(lettuce_test_feats).drop(columns=["error"], errors="ignore")
lettuce_y_train = lettuce_train_rows["label"].to_numpy()
lettuce_y_test = lettuce_test_rows["label"].to_numpy()

lettuce_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
])
lettuce_clf.fit(lettuce_train_X, lettuce_y_train)

lettuce_test_pred = lettuce_clf.predict(lettuce_test_X)
lettuce_test_score = lettuce_clf.predict_proba(lettuce_test_X)[:, 1]

test_df_lettuce = lettuce_test_rows.copy()
test_df_lettuce["lettuce_pred"] = lettuce_test_pred
test_df_lettuce["lettuce_score"] = lettuce_test_score

lettuce_overall = pd.DataFrame([
    evaluate_predictions(lettuce_y_test, lettuce_test_pred, lettuce_test_score, prefix="lettuce_supervised")
])
lettuce_by_type = evaluate_by_type(test_df_lettuce, pred_col="lettuce_pred", score_col="lettuce_score")

display(lettuce_overall)
display(lettuce_by_type)


## 8. Baseline C: improved LookBackLens features

We keep the attention-based intuition from the original notebook, but train on real labels and use stronger aggregate features:
- `mean_ratio`
- `min_ratio`
- `frac_low_03`
- `frac_low_02`
- `std_ratio`
- `bottom3_mean`
- `longest_low_streak`
- numeric-token grounding features


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "gpt2"  # Replace with a stronger causal LM if available.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading LookBackLens backbone: {MODEL_NAME} on {DEVICE}")
lbl_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
lbl_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    attn_implementation="eager",
).to(DEVICE)
lbl_model.eval()

if lbl_tokenizer.pad_token is None:
    lbl_tokenizer.pad_token = lbl_tokenizer.eos_token

print("Model loaded.")


In [ ]:
def compute_lookback_ratios(model, tokenizer, context, answer, max_context_tokens=256, max_answer_tokens=128):
    ctx_ids = tokenizer.encode(context, add_special_tokens=True, truncation=True, max_length=max_context_tokens)
    ans_ids = tokenizer.encode(answer, add_special_tokens=False, truncation=True, max_length=max_answer_tokens)
    if not ans_ids:
        return []

    context_len = len(ctx_ids)
    full_ids = ctx_ids + ans_ids
    input_ids = torch.tensor([full_ids], dtype=torch.long).to(model.device)

    with torch.no_grad():
        outputs = model(input_ids, output_attentions=True)

    n_layers = len(outputs.attentions)
    avg_attn = outputs.attentions[0][0].mean(0)
    for layer_attn in outputs.attentions[1:]:
        avg_attn = avg_attn + layer_attn[0].mean(0)
    avg_attn = (avg_attn / n_layers).cpu()

    answer_tokens = tokenizer.convert_ids_to_tokens(ans_ids)
    ratios = []
    total_len = len(full_ids)

    for offset, t in enumerate(range(context_len, total_len)):
        row = avg_attn[t, : t + 1]
        attn_ctx = row[:context_len].sum().item()
        attn_gen = row[context_len:t].sum().item()
        denom = attn_ctx + attn_gen
        ratio = (attn_ctx / denom) if denom > 1e-9 else 0.5
        token_str = answer_tokens[offset] if offset < len(answer_tokens) else "<unk>"
        ratios.append({
            "token": token_str,
            "lookback_ratio": ratio,
            "attn_to_context": attn_ctx,
            "attn_to_generated": attn_gen,
        })

    del outputs
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return ratios


def longest_streak(flags):
    best = 0
    cur = 0
    for flag in flags:
        cur = cur + 1 if flag else 0
        best = max(best, cur)
    return best


def aggregate_lookback_features(ratios):
    if not ratios:
        return {
            "mean_ratio": 0.5,
            "min_ratio": 0.5,
            "frac_low_03": 0.0,
            "frac_low_02": 0.0,
            "std_ratio": 0.0,
            "bottom3_mean": 0.5,
            "longest_low_streak": 0.0,
            "mean_ratio_numeric": 0.5,
            "frac_low_numeric": 0.0,
        }

    vals = np.array([r["lookback_ratio"] for r in ratios], dtype=float)
    bottom3 = np.sort(vals)[: min(3, len(vals))]
    numeric_mask = np.array([bool(re.search(r"\d", r["token"])) for r in ratios])

    if numeric_mask.any():
        numeric_vals = vals[numeric_mask]
        mean_ratio_numeric = float(numeric_vals.mean())
        frac_low_numeric = float((numeric_vals < 0.3).mean())
    else:
        mean_ratio_numeric = 0.5
        frac_low_numeric = 0.0

    low_flags = vals < 0.3
    return {
        "mean_ratio": float(vals.mean()),
        "min_ratio": float(vals.min()),
        "frac_low_03": float((vals < 0.3).mean()),
        "frac_low_02": float((vals < 0.2).mean()),
        "std_ratio": float(vals.std()),
        "bottom3_mean": float(bottom3.mean()),
        "longest_low_streak": float(longest_streak(low_flags.tolist())),
        "mean_ratio_numeric": mean_ratio_numeric,
        "frac_low_numeric": frac_low_numeric,
    }


In [ ]:
LBL_TRAIN_LIMIT = None
LBL_TEST_LIMIT = None

lbl_train_rows = train_df if LBL_TRAIN_LIMIT is None else train_df.iloc[:LBL_TRAIN_LIMIT]
lbl_test_rows = test_df if LBL_TEST_LIMIT is None else test_df.iloc[:LBL_TEST_LIMIT]

lbl_train_feats = []
for _, row in tqdm(lbl_train_rows.iterrows(), total=len(lbl_train_rows), desc="LookBack train"):
    context_input = f"Question: {row['query']}\nContext: {row['context']}"
    try:
        ratios = compute_lookback_ratios(lbl_model, lbl_tokenizer, context=context_input, answer=row["output"])
        feats = aggregate_lookback_features(ratios)
    except Exception as exc:
        feats = {
            "mean_ratio": 0.5,
            "min_ratio": 0.5,
            "frac_low_03": 0.0,
            "frac_low_02": 0.0,
            "std_ratio": 0.0,
            "bottom3_mean": 0.5,
            "longest_low_streak": 0.0,
            "mean_ratio_numeric": 0.5,
            "frac_low_numeric": 0.0,
            "error": str(exc),
        }
    lbl_train_feats.append(feats)

lbl_test_feats = []
for _, row in tqdm(lbl_test_rows.iterrows(), total=len(lbl_test_rows), desc="LookBack test"):
    context_input = f"Question: {row['query']}\nContext: {row['context']}"
    try:
        ratios = compute_lookback_ratios(lbl_model, lbl_tokenizer, context=context_input, answer=row["output"])
        feats = aggregate_lookback_features(ratios)
    except Exception as exc:
        feats = {
            "mean_ratio": 0.5,
            "min_ratio": 0.5,
            "frac_low_03": 0.0,
            "frac_low_02": 0.0,
            "std_ratio": 0.0,
            "bottom3_mean": 0.5,
            "longest_low_streak": 0.0,
            "mean_ratio_numeric": 0.5,
            "frac_low_numeric": 0.0,
            "error": str(exc),
        }
    lbl_test_feats.append(feats)

lbl_train_X = pd.DataFrame(lbl_train_feats).drop(columns=["error"], errors="ignore")
lbl_test_X = pd.DataFrame(lbl_test_feats).drop(columns=["error"], errors="ignore")
lbl_y_train = lbl_train_rows["label"].to_numpy()
lbl_y_test = lbl_test_rows["label"].to_numpy()

lbl_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
])
lbl_clf.fit(lbl_train_X, lbl_y_train)

lbl_test_pred = lbl_clf.predict(lbl_test_X)
lbl_test_score = lbl_clf.predict_proba(lbl_test_X)[:, 1]

test_df_lbl = lbl_test_rows.copy()
test_df_lbl["lbl_pred"] = lbl_test_pred
test_df_lbl["lbl_score"] = lbl_test_score

lbl_overall = pd.DataFrame([
    evaluate_predictions(lbl_y_test, lbl_test_pred, lbl_test_score, prefix="lookback_supervised")
])
lbl_by_type = evaluate_by_type(test_df_lbl, pred_col="lbl_pred", score_col="lbl_score")

display(lbl_overall)
display(lbl_by_type)


## 9. Soft-voting ensemble

A simple ensemble often works better than any individual detector.
We average the available sample-level probabilities.


In [ ]:
if len(lettuce_test_rows) != len(test_df) or len(lbl_test_rows) != len(test_df):
    raise ValueError("Set LETTUCE_TEST_LIMIT and LBL_TEST_LIMIT to None for the ensemble, or align the subsets manually.")

ensemble_df = test_df.copy().reset_index(drop=True)
ensemble_df["lex_score"] = lex_test_score
ensemble_df["lettuce_score"] = lettuce_test_score
ensemble_df["lbl_score"] = lbl_test_score

ensemble_df["ensemble_score"] = ensemble_df[["lex_score", "lettuce_score", "lbl_score"]].mean(axis=1)
ensemble_df["ensemble_pred"] = (ensemble_df["ensemble_score"] >= 0.5).astype(int)

ensemble_overall = pd.DataFrame([
    evaluate_predictions(
        ensemble_df["label"].to_numpy(),
        ensemble_df["ensemble_pred"].to_numpy(),
        ensemble_df["ensemble_score"].to_numpy(),
        prefix="soft_vote_ensemble",
    )
])
ensemble_by_type = evaluate_by_type(ensemble_df, pred_col="ensemble_pred", score_col="ensemble_score")

display(ensemble_overall)
display(ensemble_by_type)


## 10. Compare all methods


In [ ]:
summary = pd.concat([
    lex_overall,
    lettuce_overall,
    lbl_overall,
    ensemble_overall,
], ignore_index=True)

display(summary.sort_values("f1", ascending=False))


In [ ]:
by_type_summary = pd.concat([
    lex_by_type.assign(method="lexical_verifier"),
    lettuce_by_type.assign(method="lettuce_supervised"),
    lbl_by_type.assign(method="lookback_supervised"),
    ensemble_by_type.assign(method="soft_vote_ensemble"),
], ignore_index=True)

display(by_type_summary.sort_values(["model", "f1"], ascending=[True, False]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=summary, x="model", y="f1", ax=axes[0])
axes[0].set_title("Overall F1")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=by_type_summary, x="model", y="f1", hue="method", ax=axes[1])
axes[1].set_title("Per-type F1")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()


## 11. Error analysis helper


In [ ]:
def show_errors(df, score_col, pred_col, only_type=None, n=10):
    sub = df.copy()
    if only_type is not None:
        sub = sub[sub["corruption_type"] == only_type]

    mistakes = sub[sub["label"] != sub[pred_col]].copy()
    mistakes = mistakes.sort_values(score_col, ascending=False).head(n)

    cols = ["corruption_type", "label", pred_col, score_col, "query", "output"]
    return mistakes[cols]

show_errors(ensemble_df, score_col="ensemble_score", pred_col="ensemble_pred", n=5)


## 12. Notes

Recommended next upgrades:
- replace `gpt2` with a stronger causal LM for LookBackLens-style features
- try gradient boosting on the engineered features
- add span-level evaluation for LettuceDetect against `hallucination_labels`
- add a type-router that first predicts the likely corruption type, then dispatches to a specialized detector
